# Notebook 2: Data Collection & Cleaning
**Goal:** Download Anuvaad (en-gu) and IndicNLP Gujarati corpus, filter healthcare sentences, clean and deduplicate.

**Output:** `data/clean_gu_health.csv`

In [1]:
import os, re, csv, json, hashlib, unicodedata, requests
import pandas as pd
from tqdm import tqdm

os.makedirs('data', exist_ok=True)
print('Directories ready')

Directories ready


## 1. Download Anuvaad en-gu Dataset (HuggingFace)

In [2]:
from datasets import load_dataset

# Option A: OPUS Anuvaad via HuggingFace datasets (fastest)
print('Loading Anuvaad en-gu from HuggingFace...')
try:
    anuvaad_ds = load_dataset(
        'Helsinki-NLP/opus-100', 'en-gu',
        split='train',
        trust_remote_code=True
    )
    print(f'Anuvaad loaded: {len(anuvaad_ds):,} sentence pairs')
    print('Sample:', anuvaad_ds[0])
except Exception as e:
    print(f'HuggingFace load failed: {e}')
    print('Trying alternative source...')
    anuvaad_ds = load_dataset(
        'ai4bharat/anuvaad-parallel-corpus', 'en-gu',
        split='train',
        trust_remote_code=True
    )
    print(f'Anuvaad (alt) loaded: {len(anuvaad_ds):,} rows')

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Helsinki-NLP/opus-100' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading Anuvaad en-gu from HuggingFace...


README.md: 0.00B [00:00, ?B/s]

en-gu/test-00000-of-00001.parquet:   0%|          | 0.00/104k [00:00<?, ?B/s]

en-gu/train-00000-of-00001.parquet:   0%|          | 0.00/19.0M [00:00<?, ?B/s]

en-gu/validation-00000-of-00001.parquet:   0%|          | 0.00/107k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/318306 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Anuvaad loaded: 318,306 sentence pairs
Sample: {'translation': {'en': 'Grafton', 'gu': 'ગ્રાફટોનCity in Illinois, United States'}}


In [3]:
# Extract and save raw en-gu pairs to DataFrame
anuvaad_rows = []
for item in tqdm(anuvaad_ds, desc='Extracting Anuvaad pairs'):
    try:
        en_text = item['translation']['en'] if 'translation' in item else item.get('en', '')
        gu_text = item['translation']['gu'] if 'translation' in item else item.get('gu', '')
        if en_text and gu_text:
            anuvaad_rows.append({'en': en_text.strip(), 'gu': gu_text.strip(), 'source': 'anuvaad'})
    except Exception:
        continue

df_anuvaad = pd.DataFrame(anuvaad_rows)
df_anuvaad.to_csv('data/raw_anuvaad_en_gu.csv', index=False, encoding='utf-8')
print(f'Saved {len(df_anuvaad):,} raw Anuvaad pairs → data/raw_anuvaad_en_gu.csv')

Extracting Anuvaad pairs: 100%|██████████| 318306/318306 [00:07<00:00, 44966.69it/s]


Saved 318,306 raw Anuvaad pairs → data/raw_anuvaad_en_gu.csv


## 2. Download IndicNLP Gujarati Corpus (Monolingual)

In [4]:
import urllib.request

GU_CORPUS_URL = 'https://objectstore.e2enetworks.net/ai4b-public-nlu-nlg/v1-indiccorp/gu.txt'
GU_CORPUS_PATH = 'data/indicnlp_gu_raw.txt'
MAX_LINES = 500_000  # Download first 500k lines only (full file is huge)

if not os.path.exists(GU_CORPUS_PATH):
    print(f'Downloading IndicNLP Gujarati corpus (first {MAX_LINES:,} lines)...')
    with urllib.request.urlopen(GU_CORPUS_URL) as response:
        lines = []
        for i, line in enumerate(response):
            if i >= MAX_LINES:
                break
            lines.append(line.decode('utf-8').strip())
            if i % 50000 == 0:
                print(f'  Downloaded {i:,} lines...')
    with open(GU_CORPUS_PATH, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f'Saved {len(lines):,} lines → {GU_CORPUS_PATH}')
else:
    with open(GU_CORPUS_PATH, 'r', encoding='utf-8') as f:
        lines = f.read().splitlines()
    print(f'Already downloaded: {len(lines):,} lines')

HTTPError: HTTP Error 403: Forbidden

## 3. Healthcare Keyword Filtering

In [ ]:
# Gujarati & English healthcare keywords
HEALTH_KEYWORDS_GU = [
    # Diseases & conditions
    'ડાયાબિટ', 'ડાયાબીટ', 'રક્ત', 'blood', 'diabetes', 'heart', 'હૃદય',
    'cancer', 'કૅન્સર', 'કેન્સર', 'stroke', 'brain', 'મગજ',
    'kidney', 'કિડની', 'liver', 'liver', 'ઉચ્ચ BP', 'hypertension',
    'cholesterol', 'obesity', 'thyroid', 'asthma', 'tuberculosis', 'TB',
    'malaria', 'dengue', 'typhoid', 'hepatitis', 'pneumonia',
    # Symptoms
    'fever', 'તાવ', 'vomiting', 'ઉલ્ટી', 'pain', 'દુખાવો', 'diarrhea', 'ઝાડા',
    'cough', 'ખાંસી', 'fatigue', 'weakness', 'headache', 'માથું', 'dizziness',
    'chest', 'છાતી', 'breathing', 'shortness', 'swelling', 'rash', 'allergy',
    # Medicine & treatment
    'medicine', 'દવા', 'hospital', 'હૉસ્પિટલ', 'doctor', 'ડૉક્ટર', 'treatment',
    'સારવાર', 'surgery', 'vaccine', 'injection', 'therapy', 'diagnosis',
    'prescription', 'dosage', 'antibiotic', 'paracetamol', 'insulin',
    # Body parts
    'body', 'bone', 'teeth', 'eye', 'ear', 'skin', 'lung', 'stomach', 'anosdil',
    'nerve', 'immune', 'blood pressure', 'sugar', 'ખાંડ', 'weight',
]

HEALTH_KEYWORDS_EN = [
    'disease', 'illness', 'patient', 'symptom', 'diagnosis', 'treatment',
    'hospital', 'doctor', 'medicine', 'drug', 'healthcare', 'infection',
    'fever', 'pain', 'surgery', 'vaccine', 'blood', 'heart', 'diabetes',
    'cancer', 'kidney', 'liver', 'lung', 'therapy', 'clinical', 'medical',
    'health', 'cure', 'disorder', 'chronic', 'acute', 'prescription', 'dose'
]

def is_health_related(text: str, keywords: list) -> bool:
    text_lower = text.lower()
    return any(kw.lower() in text_lower for kw in keywords)

print(f'Filtering with {len(HEALTH_KEYWORDS_GU)} Gujarati + {len(HEALTH_KEYWORDS_EN)} English keywords')

## 4. Text Cleaning Functions

In [ ]:
import unicodedata

def normalize_gujarati(text: str) -> str:
    """Normalize Gujarati text: Unicode NFC, remove control chars, fix whitespace."""
    # Unicode normalization (NFC for Gujarati)
    text = unicodedata.normalize('NFC', text)
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    # Remove URLs
    text = re.sub(r'https?://\S+', '', text)
    # Remove control characters except newline
    text = ''.join(c for c in text if unicodedata.category(c) != 'Cc' or c == '\n')
    # Remove non-Gujarati non-ASCII (keep Gujarati unicode block + common punctuation + digits)
    text = re.sub(r'[^\u0A00-\u0AFF\u0020-\u007E\n।,;:.!?0-9 ]', ' ', text)
    # Collapse multiple spaces
    text = re.sub(r' {2,}', ' ', text).strip()
    return text

def get_hash(text: str) -> str:
    """MD5 hash for deduplication."""
    return hashlib.md5(text.encode('utf-8')).hexdigest()

def is_gujarati_text(text: str, threshold: float = 0.2) -> bool:
    """Check if text contains enough Gujarati characters."""
    gu_chars = sum(1 for c in text if '\u0A80' <= c <= '\u0AFF')
    return gu_chars / max(len(text), 1) >= threshold

def is_valid_sentence(text: str, min_len: int = 10, max_len: int = 512) -> bool:
    """Filter out too short or too long sentences."""
    return min_len <= len(text) <= max_len

print('Cleaning functions defined')

## 5. Process Anuvaad Dataset

In [ ]:
seen_hashes = set()
clean_records = []

df_raw = pd.read_csv('data/raw_anuvaad_en_gu.csv', encoding='utf-8')
print(f'Processing {len(df_raw):,} Anuvaad pairs...')

for _, row in tqdm(df_raw.iterrows(), total=len(df_raw), desc='Anuvaad'):
    en_text = str(row.get('en', '')).strip()
    gu_text = str(row.get('gu', '')).strip()

    # Filter: must be health-related in either language
    if not (is_health_related(en_text, HEALTH_KEYWORDS_EN) or
            is_health_related(gu_text, HEALTH_KEYWORDS_GU)):
        continue

    # Clean Gujarati text
    gu_clean = normalize_gujarati(gu_text)

    # Validate
    if not is_gujarati_text(gu_clean): continue
    if not is_valid_sentence(gu_clean): continue

    # Deduplicate
    h = get_hash(gu_clean)
    if h in seen_hashes: continue
    seen_hashes.add(h)

    clean_records.append({
        'gu': gu_clean,
        'en': en_text,
        'source': 'anuvaad'
    })

print(f'Anuvaad: {len(clean_records):,} clean healthcare sentences after filtering')

## 6. Process IndicNLP Gujarati Corpus

In [ ]:
indicnlp_count = 0
with open(GU_CORPUS_PATH, 'r', encoding='utf-8') as f:
    corpus_lines = f.read().splitlines()

print(f'Processing {len(corpus_lines):,} IndicNLP lines...')

for line in tqdm(corpus_lines, desc='IndicNLP'):
    if not is_health_related(line, HEALTH_KEYWORDS_GU):
        continue

    gu_clean = normalize_gujarati(line)

    if not is_gujarati_text(gu_clean): continue
    if not is_valid_sentence(gu_clean): continue

    h = get_hash(gu_clean)
    if h in seen_hashes: continue
    seen_hashes.add(h)

    clean_records.append({'gu': gu_clean, 'en': '', 'source': 'indicnlp'})
    indicnlp_count += 1

print(f'IndicNLP: {indicnlp_count:,} additional healthcare sentences added')

## 7. Save Clean Dataset

In [ ]:
df_clean = pd.DataFrame(clean_records)
df_clean = df_clean.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

output_path = 'data/clean_gu_health.csv'
df_clean.to_csv(output_path, index=False, encoding='utf-8')

print('\nDATASET SUMMARY')
print('='*50)
print(f'Total sentences:   {len(df_clean):,}')
print(f'Source breakdown:')
print(df_clean['source'].value_counts().to_string())
print(f'\nWith English pair: {df_clean["en"].str.len().gt(0).sum():,}')
print(f'Avg Gujarati len:  {df_clean["gu"].str.len().mean():.0f} chars')
print(f'\nSaved → {output_path}')
print('\nNEXT STEP: Run Notebook 03 to build Knowledge Graph.')

# Show samples
print('\n--- Sample Records ---')
print(df_clean[['gu', 'source']].sample(5).to_string())